# Pair tensors → PDBs (original + reconstructed)

Discovers proteins from **`reconstructed_pair_npy/*_reconstructed_pair.npy`**, then:

1. **`pdbs/original/`** — run `run_structure_module.py` on **`proteins_layer47/<id>/<id>_pair_block_47.npy`** (same proteins).
2. **`pdbs/reconstructed/`** — same script on each **`reconstructed_pair.npy`**.

Default roots assume this notebook lives in **`AlphaFold_Autoencoder/original_SAE/`** (sibling folders: `reconstructed_pair_npy/`, `proteins_layer47/`, `pdbs/`). Override with env vars in the config cell.

**OpenFold path:** defaults to **`…/AlphaFold_Autoencoder/openfold`** (the **clone root** — the directory that contains the inner **`openfold/`** package folder). This notebook adds it to `sys.path` and sets **`PYTHONPATH` for subprocesses** (child processes do not see `sys.path`). Override with env **`OPENFOLD_REPO`**. Also requires `run_structure_module.py` next to this notebook; optional FASTA per protein subfolder.

**Slow runs:** Each protein spawns a **new Python process** that **imports OpenFold and loads weights** — the first minute+ per protein can look “stuck”. **`KeyboardInterrupt`** usually means you stopped the cell while waiting; let it run or test on **one** protein first.

**`AlphaFold` import errors** in the first code cell: usually **missing deps** or a **broken `attention_core.py`** (see troubleshooting printed there). Install from OpenFold’s `requirements.txt` if needed.

**Setup (first code cell):** sets `sys.path`, **`PYTHONPATH`** (required for subprocesses), and runs import checks. **`IndentationError` in `attention_core.py`** means that file in your OpenFold clone is broken—restore it from git or upstream (see traceback hint).


In [ ]:
from __future__ import annotations

import glob
import os
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np


def _find_script_dir() -> Path:
    p = Path.cwd().resolve()
    for _ in range(10):
        if (p / "run_structure_module.py").is_file():
            return p
        p = p.parent
    return Path.cwd().resolve()


SCRIPT_DIR = _find_script_dir()
AUTOENCODER_ROOT = Path(os.environ.get("AUTOENCODER_ROOT", str(SCRIPT_DIR.parent))).resolve()

# OpenFold clone root = folder containing inner package `openfold/` (override: env OPENFOLD_REPO)
OPENFOLD_REPO = Path(os.environ.get("OPENFOLD_REPO", str(AUTOENCODER_ROOT / "openfold"))).resolve()

if OPENFOLD_REPO.is_dir() and (OPENFOLD_REPO / "openfold").is_dir():
    if str(OPENFOLD_REPO) not in sys.path:
        sys.path.insert(0, str(OPENFOLD_REPO))
    # Subprocesses do not see sys.path — run_structure_module.py needs PYTHONPATH
    _old_pp = os.environ.get("PYTHONPATH", "")
    os.environ["PYTHONPATH"] = str(OPENFOLD_REPO) + (os.pathsep + _old_pp if _old_pp else "")
    print("OPENFOLD_REPO:", OPENFOLD_REPO)
    print("PYTHONPATH starts with OPENFOLD_REPO (for subprocesses)")
else:
    print(
        "WARNING: OpenFold not found at",
        OPENFOLD_REPO,
        "— expected .../openfold/openfold/ (package). Set env OPENFOLD_REPO.",
    )

# --- Same imports as run_structure_module.py (kernel + subprocess must both work) ---
try:
    import openfold  # noqa: F401

    print("import openfold OK ->", Path(openfold.__file__).resolve())
except ModuleNotFoundError as e:
    print("import openfold FAILED:", e)

try:
    from openfold.config import model_config
    from openfold.model.model import AlphaFold

    print("openfold.config + AlphaFold model import OK")
except Exception:
    import traceback

    traceback.print_exc()
    print(
        "\n--- Troubleshooting ---\n"
        "• IndentationError in attention_core.py: corrupted file in your clone. Fix with:\n"
        "    cd $OPENFOLD_REPO && git checkout -- openfold/utils/kernel/attention_core.py\n"
        "  or re-download from:\n"
        "    https://raw.githubusercontent.com/aqlaboratory/openfold/main/openfold/utils/kernel/attention_core.py\n"
        "• ModuleNotFoundError: install OpenFold deps (requirements.txt) in this env.\n"
        "• attn_core_inplace_cuda missing: build/install OpenFold with CUDA (pip install -e .).\n"
    )

print("AUTOENCODER_ROOT:", AUTOENCODER_ROOT)
print("SCRIPT_DIR:", SCRIPT_DIR)


In [ ]:
# --- Paths (override with env) ---
RECON_NPY_DIR = Path(
    os.environ.get("RECONSTRUCTED_NPY_DIR", str(AUTOENCODER_ROOT / "reconstructed_pair_npy"))
).resolve()
PROTEIN_DATA_ROOT = Path(
    os.environ.get("PROTEIN_DATA_ROOT", str(AUTOENCODER_ROOT / "proteins_layer47"))
).resolve()
OUT_ORIGINAL = Path(os.environ.get("PDB_ORIGINAL_DIR", str(AUTOENCODER_ROOT / "pdbs" / "original"))).resolve()
OUT_RECON = Path(os.environ.get("PDB_RECON_DIR", str(AUTOENCODER_ROOT / "pdbs" / "reconstructed"))).resolve()

RUN_STRUCTURE = Path(os.environ.get("RUN_STRUCTURE_MODULE", str(SCRIPT_DIR / "run_structure_module.py")))
LAYER = int(os.environ.get("PAIR_LAYER", "47"))
MODEL_DEVICE = os.environ.get("STRUCTURE_MODEL_DEVICE", "cuda" if __import__("torch").cuda.is_available() else "cpu")
TIMEOUT_SEC = int(os.environ.get("STRUCTURE_TIMEOUT", "7200"))  # large structures can be slow

# OpenFold weights (required by run_structure_module.py)
_ckpt_env = os.environ.get("OPENFOLD_CHECKPOINT_PATH")
if _ckpt_env:
    OPENFOLD_CHECKPOINT_PATH = Path(_ckpt_env).resolve()
else:
    OPENFOLD_CHECKPOINT_PATH = (OPENFOLD_REPO / "openfold/resources/openfold_params/finetuning_ptm_1.pt").resolve()
if not OPENFOLD_CHECKPOINT_PATH.is_file():
    OPENFOLD_CHECKPOINT_PATH = None  # type: ignore

OUT_ORIGINAL.mkdir(parents=True, exist_ok=True)
OUT_RECON.mkdir(parents=True, exist_ok=True)

print("RECON_NPY_DIR:", RECON_NPY_DIR)
print("PROTEIN_DATA_ROOT:", PROTEIN_DATA_ROOT)
print("OUT_ORIGINAL:", OUT_ORIGINAL)
print("OUT_RECON:", OUT_RECON)
print("MODEL_DEVICE:", MODEL_DEVICE)
if OPENFOLD_CHECKPOINT_PATH is not None:
    print("OPENFOLD_CHECKPOINT_PATH:", OPENFOLD_CHECKPOINT_PATH)
else:
    print(
        "WARNING: No checkpoint found. Set env OPENFOLD_CHECKPOINT_PATH=/path/to/finetuning_ptm_1.pt "
        "or download into OPENFOLD_REPO/openfold/resources/openfold_params/ (see OpenFold docs)."
    )

In [ ]:
def find_fasta(subdir: Path, protein_id: str) -> Path | None:
    for fn in (f"{protein_id}.fasta", "seq.fasta"):
        p = subdir / fn
        if p.is_file():
            return p
    for f in subdir.iterdir():
        if f.suffix == ".fasta":
            return f
    return None


def find_original_pair_npy(protein_id: str, protein_root: Path, layer: int) -> Path | None:
    subdir = protein_root / protein_id
    if not subdir.is_dir():
        return None
    for name in (f"{protein_id}_pair_block_{layer}.npy", f"{protein_id}_pair_block{layer}.npy"):
        p = subdir / name
        if p.is_file():
            return p
    globs = list(subdir.glob("*pair_block*47*.npy")) + list(subdir.glob("*pair_block47*.npy"))
    return sorted(globs, key=lambda x: x.name)[0] if globs else None


def run_structure_on_pair(
    pair_npy: Path,
    out_dir: Path,
    protein_id: str,
) -> tuple[bool, str]:
    """Invoke run_structure_module.py; return (ok, message)."""
    if not RUN_STRUCTURE.is_file():
        return False, f"Missing {RUN_STRUCTURE}"
    if OPENFOLD_CHECKPOINT_PATH is None:
        return (
            False,
            "No checkpoint: set OPENFOLD_CHECKPOINT_PATH or place finetuning_ptm_1.pt in "
            "OPENFOLD_REPO/openfold/resources/openfold_params/",
        )

    pair = np.load(pair_npy, allow_pickle=True)
    n_res = int(pair.shape[0]) if pair.ndim >= 2 else int(pair.shape[1])

    cmd = [
        sys.executable,
        str(RUN_STRUCTURE),
        str(pair_npy),
        str(out_dir),
        "--single-from-pair",
        "--model-device",
        MODEL_DEVICE,
    ]
    if OPENFOLD_CHECKPOINT_PATH is not None:
        cmd.extend(["--openfold-checkpoint-path", str(OPENFOLD_CHECKPOINT_PATH)])

    subdir = PROTEIN_DATA_ROOT / protein_id
    fasta_path = find_fasta(subdir, protein_id) if subdir.is_dir() else None
    temp_aatype: str | None = None
    try:
        if fasta_path:
            cmd.extend(["--fasta", str(fasta_path)])
        else:
            fd, temp_aatype = tempfile.mkstemp(suffix=".npy")
            os.close(fd)
            np.save(temp_aatype, np.zeros(n_res, dtype=np.int64))
            cmd.extend(["--aatype-npy", temp_aatype])

        env = os.environ.copy()
        if (OPENFOLD_REPO / "openfold").is_dir():
            _old = env.get("PYTHONPATH", "")
            env["PYTHONPATH"] = str(OPENFOLD_REPO) + (os.pathsep + _old if _old else "")
        # CWD must be clone root so relative paths like openfold/resources/... work
        r = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=TIMEOUT_SEC,
            env=env,
            cwd=str(OPENFOLD_REPO),
        )
        stem = Path(pair_npy).stem
        expected = out_dir / f"{stem}_structure.pdb"
        if r.returncode == 0 and expected.is_file():
            return True, str(expected)
        err = (r.stderr or r.stdout or "").strip()
        if len(err) > 6000:
            err = err[:6000] + "\n... [truncated]"
        return False, err
    except subprocess.TimeoutExpired:
        return False, f"timeout after {TIMEOUT_SEC}s"
    except Exception as e:
        return False, repr(e)
    finally:
        if temp_aatype and os.path.exists(temp_aatype):
            try:
                os.unlink(temp_aatype)
            except OSError:
                pass

In [ ]:
pattern = str(RECON_NPY_DIR / "*_reconstructed_pair.npy")
recon_files = sorted(glob.glob(pattern))
if not recon_files:
    raise FileNotFoundError(f"No *_reconstructed_pair.npy under {RECON_NPY_DIR}")

protein_ids = [Path(p).stem.replace("_reconstructed_pair", "") for p in recon_files]
print(f"Found {len(protein_ids)} proteins from reconstructed npy list.")

In [ ]:
# 1) Original pair_block → pdbs/original/
ok_o = fail_o = 0
for pid in protein_ids:
    orig = find_original_pair_npy(pid, PROTEIN_DATA_ROOT, LAYER)
    if orig is None:
        print(f"[original] {pid} -> SKIP (no pair_block npy under {PROTEIN_DATA_ROOT / pid})")
        fail_o += 1
        continue
    good, msg = run_structure_on_pair(orig, OUT_ORIGINAL, pid)
    if good:
        print(f"[original] {pid} -> OK {msg}")
        ok_o += 1
    else:
        print(f"[original] {pid} -> FAIL\n{msg}")
        fail_o += 1

print(f"\nOriginal done: {ok_o} ok, {fail_o} failed/skipped")

In [ ]:
# 2) Reconstructed npy → pdbs/reconstructed/
ok_r = fail_r = 0
for npy_path in recon_files:
    pid = Path(npy_path).stem.replace("_reconstructed_pair", "")
    good, msg = run_structure_on_pair(Path(npy_path), OUT_RECON, pid)
    if good:
        print(f"[recon] {pid} -> OK {msg}")
        ok_r += 1
    else:
        print(f"[recon] {pid} -> FAIL\n{msg}")
        fail_r += 1

print(f"\nReconstructed done: {ok_r} ok, {fail_r} failed")